In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import pandas as pd
import random
import math
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# shared, de-duplicated implementations for the semi-supervised family
from src.semi import (
    TextDataset, DataFactory, mmd_loss,
    Classifier, BERTClassifier, RoBERTaClassifier, LSTM_BERT_Classifier,
    SemiMain,
)


In [2]:
class Main(SemiMain):

    def train_on_cls_embeddings(self, epochs=15, batch_size=32, lr=1e-4):
            """
            Fine-tunes only the classifier head using precomputed [CLS] embeddings.
            Args:
                dataset (SMOTEVectorDataset): Dataset of CLS embeddings, labels, and IDs.
                epochs (int): Number of training epochs.
                batch_size (int): Training batch size.
                lr (float): Learning rate.
            """
            model = self.model
            device = self.device
            min_loss = math.inf
            patience = 3
            model.to(device)
            model.train()

            # Freeze non-head parts
            for p in model.bert.parameters():
                p.requires_grad = False
            for p in model.proj.parameters():
                p.requires_grad = False

            # DataLoader from given dataset
            train_loader = self.second_trainloader

            # Only train fc and classifier
            params = list(model.fc.parameters()) + list(model.classifier.parameters())
            optimizer = torch.optim.Adam(params, lr=lr)
            criterion = nn.CrossEntropyLoss()

            for epoch in range(self.num_epochs):
                self.model.train()
                epoch_loss = []
                epoch_acc = []
                epoch_f1 = []

                for cls_vec, y, _ in tqdm(train_loader):  # IDs are ignored
                    cls_vec, y = cls_vec.to(device), y.to(device)
                    
                    optimizer.zero_grad()
                    cls_out = model.fc(cls_vec)
                    outputs = model.classifier(cls_out + cls_vec)
                    pred = torch.argmax(outputs, dim=1)
                    loss_ce = criterion(outputs, y)
                    loss_supcon = self.supcon_loss(outputs, y)
                    loss = loss_ce
                    loss.backward()
                    optimizer.step()
                    epoch_loss.append(loss.item())
                    epoch_acc.append(self.metric1(pred.detach().cpu(), y.detach().cpu()))
                    epoch_f1.append(self.metric2(pred.detach().cpu(), y.detach().cpu(), average='macro'))

                epoch_loss = np.mean(epoch_loss)
                epoch_acc = np.mean(epoch_acc)
                epoch_f1 = np.mean(epoch_f1)
                print(f"Epoch {epoch + 1:>3}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}, F1: {epoch_f1:.4f}")

                vali_loss = self.second_validation()
                self.model.train()
                if vali_loss < min_loss:
                    min_loss = vali_loss
                    self.__save__()
                else:
                    patience -= 1

                if not patience:
                    break


In [3]:
class Config:
    lambda_mmd = 0.001
    name = 'BERT_SMOTE_semi'
    embed_dim = 256
    lstm_hidden_dim = 64
    num_heads = 16
    hidden_dim = 128
    num_layers = 2
    num_classes = 2
    path1 = '../data/raw/domain1_train_data.json'
    path2 = '../data/raw/domain2_train_data.json'
    test_path = '../data/raw/test_data.json'
    num_epochs = 15
    batch_size1 = 32
    batch_size2 = 32
    seed = 42
    tau=1
    save_path = "../results/SMOTE_result.csv"

configs = Config()
main = Main(configs)

In [4]:
# --- Data layers: landing (merged intermediate) + curated (cleaned) ---
import os
import pandas as pd

_d1 = pd.read_json(configs.path1, lines=True); _d1['domain'] = 0
_d2 = pd.read_json(configs.path2, lines=True); _d2['domain'] = 1
_merged = pd.concat(
    [_d1[['id', 'text', 'label', 'domain']], _d2[['id', 'text', 'label', 'domain']]],
    ignore_index=True,
)

# Landing: domain1 + domain2 merged, before any cleaning (raw lineage)
os.makedirs('../data/landing', exist_ok=True)
_merged.to_json('../data/landing/domain_merged.json', orient='records', lines=True)
print(f"landing/domain_merged.json  <- {len(_merged)} rows")

# Curated: drop empty-text rows, unified {id, text, label, domain} schema
_curated = _merged[_merged['text'].map(len) > 0].reset_index(drop=True)
os.makedirs('../data/curated', exist_ok=True)
_curated.to_json('../data/curated/train_clean.json', orient='records', lines=True)
print(f"curated/train_clean.json    <- {len(_curated)} rows "
      f"(dropped {len(_merged) - len(_curated)} empty-text rows)")


landing/domain_merged.json  <- 6000 rows
curated/train_clean.json    <- 6000 rows (dropped 0 empty-text rows)


In [5]:
def save_pseudo_to_json(pseudo_samples, path="../data/landing/pseudo_labeled.json"):
    import os
    os.makedirs(os.path.dirname(path), exist_ok=True)
    serializable_data = []
    # collect_pseudo_labels yields (text, label, id) tuples
    for text, label, id_ in pseudo_samples:
        serializable_data.append({
            "id": int(id_),
            "label": int(label),
            "text": text.tolist(),
        })

    with open(path, "w") as f:
        for entry in serializable_data:
            f.write(json.dumps(entry) + "\n")

    print(f"Pseudo-labeled data saved to {path} ({len(serializable_data)} samples)")


In [6]:
def extract_raw_data(dataset):
    result = []
    for i in range(len(dataset)):
        text, label, idx = dataset[i]
        mask = (text != 0).long()
        result.append((text, label, idx))
    return result

In [7]:
def build_dataloader_from_pseudo(data_list, batch_size, collate_fn):
    class PseudoDataset(torch.utils.data.Dataset):
        def __init__(self, data):
            self.data = data  # (text, mask, label, id)

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            text, label, idx_ = self.data[idx]
            return text, label, idx_  
    dataset = PseudoDataset(data_list)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    return loader


In [8]:
class SMOTEVectorDataset(Dataset):
    def __init__(self, features, labels, ids):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.ids = ids

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx], torch.tensor(self.ids[idx], dtype=torch.long)


In [9]:
main.train()

0it [00:00, ?it/s]

1it [00:00,  3.18it/s]

2it [00:00,  3.29it/s]

3it [00:00,  3.40it/s]

4it [00:01,  3.44it/s]

...

6it [00:00, 12.77it/s]

8it [00:00, 12.47it/s]

10it [00:00, 12.94it/s]

10it [00:00, 12.86it/s]

Validation Loss: 0.2974, Accuracy: 0.9141, F1: 0.8810


In [10]:
from imblearn.over_sampling import SMOTE
from torch.utils.data import Dataset, DataLoader

pseudo_data_1, pseudo_data_2, val_data_1, val_data_2 = main.collect_pseudo_labels(threshold=0.99)
train_data_1 = extract_raw_data(main.trainloader_1.dataset)
train_data_2 = extract_raw_data(main.trainloader_2.dataset)

combined_train_data_1 = train_data_1 + pseudo_data_1
combined_train_data_2 = train_data_2 + pseudo_data_2
temp_loader_1 = build_dataloader_from_pseudo(combined_train_data_1, batch_size=32, collate_fn=main.data2.collate_fn)
temp_loader_2 = build_dataloader_from_pseudo(combined_train_data_2, batch_size=32, collate_fn=main.data2.collate_fn)

feat_1, label_1, id_1 = main.model.extract_cls_from_tokens(temp_loader_1, main.device)
feat_2, label_2, id_2 = main.model.extract_cls_from_tokens(temp_loader_2, main.device)

features = torch.cat([feat_1, feat_2])
labels = torch.cat([label_1, label_2])
ids = id_1 + id_2

sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(features.numpy(), labels.numpy())
id_resampled = ids + [999999 + i for i in range(len(X_resampled) - len(ids))]
smote_dataset = SMOTEVectorDataset(X_resampled, y_resampled, id_resampled)

# Replace loaders for phase 2 training
main.second_trainloader = DataLoader(smote_dataset, batch_size=main.configs.batch_size1, shuffle=True)

val_data = val_data_1 + val_data_2
# Build validation loaders from remaining validation data
val_loader = build_dataloader_from_pseudo(val_data, batch_size=main.configs.batch_size1, collate_fn=main.data2.collate_fn)

# Assign to main class
main.second_valloader = val_loader

main.train_on_cls_embeddings()  # Phase 2 training with updated loaders



0it [00:00, ?it/s]

2it [00:00, 12.12it/s]

4it [00:00, 11.91it/s]

6it [00:00, 12.07it/s]

8it [00:00, 12.36it/s]

...

 85%|████████▌ | 217/254 [00:00<00:00, 235.94it/s]

 95%|█████████▍| 241/254 [00:01<00:00, 236.48it/s]

100%|██████████| 254/254 [00:01<00:00, 232.26it/s]

Epoch   6, Loss: 0.0148, Accuracy: 0.9959, F1: 0.9957


[Second Val] Loss: 0.1855, Acc: 0.8924, F1: 0.8865


In [11]:
# --- Landing: persist the high-confidence pseudo-labeled samples ---
save_pseudo_to_json(pseudo_data_1 + pseudo_data_2)


Pseudo-labeled data saved to ../data/landing/pseudo_labeled.json (419 samples)


In [12]:
from imblearn.over_sampling import SMOTE
from torch.utils.data import Dataset, DataLoader

class PseudoDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data  # each item should be (text, label, id) or (text, mask, label, id)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if len(item) == 4:
            return item  # (text, mask, label, id)
        elif len(item) == 3:
            text, label, idx_ = item
            mask = (text != 0).long()
            return text, mask, label, idx_
        else:
            raise ValueError(f"Expected 3 or 4 values, got {len(item)} at index {idx}")

def extract_raw_data(dataset):
    return [dataset[i] for i in range(len(dataset))]  # (text, label, id[, domain])

def build_phase2_loaders_with_smote(main, pseudo_data):
    # Combine all samples
    train_data_1 = extract_raw_data(main.trainloader_1.dataset)
    train_data_2 = extract_raw_data(main.trainloader_2.dataset)
    combined_data = train_data_1 + train_data_2 + pseudo_data
    for i in range(len(combined_data)):
        if len(combined_data[i]) == 3:
            text, label, idx = combined_data[i]
            mask = (text != 0).long()
            combined_data[i] = (text, mask, label, idx)

    # Build a temporary DataLoader from combined_data
    temp_loader = build_dataloader_from_pseudo(combined_data, batch_size=32, collate_fn=main.data2.simple_collate_fn)

    # Extract semantic features using the BERT classifier
    features, labels, ids = main.model.extract_cls_from_tokens(temp_loader, main.device)

    # Apply SMOTE to semantic features
    sm = SMOTE(random_state=42)
    X_resampled, y_resampled = sm.fit_resample(features.numpy(), labels.numpy())
    id_resampled = ids + [999999 + i for i in range(len(X_resampled) - len(ids))]

    # Build Dataset and DataLoaders
    smote_dataset = SMOTEVectorDataset(X_resampled, y_resampled, id_resampled)
    mid = len(smote_dataset) // 2
    dataset1 = torch.utils.data.Subset(smote_dataset, list(range(0, mid)))
    dataset2 = torch.utils.data.Subset(smote_dataset, list(range(mid, len(smote_dataset))))

    # Replace loaders for phase 2 training
    main.trainloader_1 = DataLoader(dataset1, batch_size=main.configs.batch_size1, shuffle=True)
    main.trainloader_2 = DataLoader(dataset2, batch_size=main.configs.batch_size2, shuffle=True)

    # Dataset from embeddings
    class CLSEmbeddingDataset(Dataset):
        def __init__(self, features, labels):
            self.features = torch.tensor(features, dtype=torch.float32)
            self.labels = torch.tensor(labels, dtype=torch.long)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return self.features[idx], self.labels[idx]


In [13]:
train_data_1 = extract_raw_data(main.trainloader_1.dataset)
train_data_2 = extract_raw_data(main.trainloader_2.dataset)
val_data_1 = extract_raw_data(main.valloader_1.dataset)
val_data_2 = extract_raw_data(main.valloader_2.dataset)
full_data = train_data_1 + train_data_2 + val_data_1 + val_data_2
full_loader = build_dataloader_from_pseudo(full_data, configs.batch_size2, main.data2.collate_fn)
def evaluate_model(model, dataloader, criterion, device, source_name="all"):
    model.eval()
    all_preds, all_labels, all_ids = [], [], []
    misclassified = []

    total_loss = 0

    with torch.no_grad():
        for x, x_mask, y, ids in dataloader:
            x, x_mask, y = x.to(device), x_mask.to(device), y.to(device)
            outputs, _ = model(x, x_mask)
            loss = criterion(outputs, y)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_ids.extend(ids.cpu().numpy())

            for i in range(len(y)):
                true_label = y[i].item()
                pred_label = preds[i].item()
                sample_id = ids[i].item()
                if true_label != pred_label:
                    misclassified.append((sample_id, true_label, pred_label, source_name))

    from sklearn.metrics import accuracy_score, f1_score
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    print(f"\n Evaluation on {source_name} — Loss: {total_loss:.4f}, Accuracy: {acc:.4f}, F1: {f1:.4f}")
    return
evaluate_model(main.model, full_loader, main.criterion, main.device)




 Evaluation on all — Loss: 13.8175, Accuracy: 0.9825, F1: 0.9599


In [14]:
main.test()

  0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 17/4000 [00:00<00:23, 168.31it/s]

  1%|          | 34/4000 [00:00<00:23, 165.40it/s]

  1%|▏         | 51/4000 [00:00<00:37, 105.97it/s]

  2%|▏         | 64/4000 [00:00<00:37, 105.77it/s]

...

 99%|█████████▉| 3954/4000 [00:32<00:00, 140.04it/s]

 99%|█████████▉| 3971/4000 [00:32<00:00, 146.27it/s]

100%|█████████▉| 3988/4000 [00:33<00:00, 145.61it/s]

100%|██████████| 4000/4000 [00:33<00:00, 120.48it/s]

Saved -> ../results/SMOTE_result.csv
